# QQQ 시작연도별 인출 백테스트

투자 조건을 입력하고 아래 코드 셀을 실행하세요. 시작연도별 생존 여부와 종료 잔액을 계산하고, 요약 CSV·월별 상세 CSV·PNG 그래프를 생성합니다.


In [ ]:
# @title 백테스트 조건을 입력하고 실행하세요
투자종목 = "QQQ"  # @param {type:"string"}
최초_시작연도 = 2002  # @param {type:"integer"}
마지막_시작연도 = 2016  # @param {type:"integer"}
투자기간_년 = 10  # @param {type:"integer", min:1, step:1}
초기준비금_억원 = 1.0  # @param {type:"number", min:0.01, step:0.1}
월인출액_만원 = 100  # @param {type:"number", min:0, step:10}

"""미국 ETF의 시작연도별 정기 인출 결과를 비교한다.

선택 종목 수익률은 Yahoo Finance 수정주가(배당·주식분할 반영),
원/달러 환율은 미국 연준 FRED DEXKOUS를 사용한다. 세금,
환전·거래 수수료와 물가상승률은 반영하지 않는다.

초기 준비금을 시작 시점 환율로 달러 환산해 선택 종목에 투자하고,
매월 말 당시 환율을 적용해 입력한 원화 금액만큼 인출한다.
"""

import os
from pathlib import Path
import subprocess
import sys
from typing import cast
from urllib.request import urlretrieve

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import font_manager
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.patches import FancyBboxPatch, Patch
import pandas as pd


def is_colab_runtime() -> bool:
    """현재 코드가 Google Colab에서 실행 중인지 확인한다."""
    return bool(os.environ.get("COLAB_RELEASE_TAG")) or Path("/content").exists()


IS_COLAB = is_colab_runtime()
OUTPUT_DIR = Path("/content/output") if IS_COLAB else Path("output")
WON_PER_EOK = 100_000_000
WON_PER_MANWON = 10_000
TICKER = str(투자종목).strip().upper()
FIRST_START_YEAR = int(최초_시작연도)
LAST_START_YEAR = int(마지막_시작연도)
INVESTMENT_YEARS = int(투자기간_년)
INITIAL_RESERVE_KRW = float(초기준비금_억원) * WON_PER_EOK
MONTHLY_WITHDRAWAL_KRW = float(월인출액_만원) * WON_PER_MANWON

try:
    import yfinance as yf
except ImportError:
    if IS_COLAB:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "yfinance"]
        )
        import yfinance as yf
    else:
        raise


# =============================================================================
# 1. 데이터 수집
# =============================================================================


def download_monthly_series(
    ticker: str,
    first_start_year: int,
    last_start_year: int,
    years: int,
) -> pd.Series:
    """Yahoo Finance에서 전년도 12월을 포함한 월말 가격을 내려받는다."""
    start = f"{first_start_year - 1}-12-01"
    # yfinance의 종료일은 포함되지 않으므로 필요한 기간보다 넉넉하게 설정한다.
    end = f"{last_start_year + years + 1}-01-01"
    raw = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        actions=False,
    )
    if raw is None or raw.empty:
        raise RuntimeError(f"{ticker} 가격 데이터를 내려받지 못했습니다.")

    series = raw["Adj Close"]
    if isinstance(series, pd.DataFrame):
        frame = cast(pd.DataFrame, series)
        series = frame[ticker] if ticker in frame.columns else frame.iloc[:, 0]
    series = series.dropna().astype(float)
    monthly = series.resample("ME").last()
    monthly.name = ticker
    return monthly


def download_monthly_usdkrw(
    first_start_year: int, last_start_year: int, years: int
) -> pd.Series:
    """미국 연준 FRED에서 월말 원/달러 환율을 내려받는다."""
    start = f"{first_start_year - 1}-12-01"
    end = f"{last_start_year + years}-12-31"
    url = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id=DEXKOUS&cosd={start}&coed={end}"
    )
    raw = pd.read_csv(url)
    date_column = "observation_date" if "observation_date" in raw.columns else "DATE"
    raw[date_column] = pd.to_datetime(raw[date_column])
    values = pd.to_numeric(raw["DEXKOUS"], errors="coerce")
    daily = pd.Series(values.to_numpy(), index=raw[date_column], name="usdkrw").dropna()
    return daily.resample("ME").last()


def load_market_data(
    ticker: str, first_start_year: int, last_start_year: int, years: int
) -> tuple[pd.Series, pd.Series]:
    """백테스트에 필요한 수정주가와 원/달러 환율을 준비한다."""
    prices = download_monthly_series(
        ticker, first_start_year, last_start_year, years
    )
    fx_rates = download_monthly_usdkrw(first_start_year, last_start_year, years)
    return prices, fx_rates


# =============================================================================
# 2. 백테스트 계산
# =============================================================================


def simulate_one_start(
    monthly_prices: pd.Series,
    monthly_fx: pd.Series,
    start_year: int,
    initial_reserve: float,
    monthly_withdrawal: float,
    years: int,
) -> tuple[dict, pd.DataFrame]:
    """월 수익률을 적용한 뒤 매월 말 정해진 원화 금액을 인출한다."""
    first_month = pd.Timestamp(start_year, 1, 31)
    previous_month = first_month - pd.offsets.MonthEnd(1)
    final_month = first_month + pd.offsets.MonthEnd(years * 12 - 1)

    required = monthly_prices.loc[previous_month:final_month]
    required_fx = monthly_fx.loc[previous_month:final_month]
    expected_count = years * 12 + 1
    if len(required) != expected_count:
        raise RuntimeError(
            f"{start_year}년: 월말 가격 {expected_count}개가 필요하지만 "
            f"{len(required)}개만 확인됐습니다."
        )
    if len(required_fx) != expected_count:
        raise RuntimeError(
            f"{start_year}년: 환율 데이터 {expected_count}개가 필요하지만 "
            f"{len(required_fx)}개만 확인됐습니다."
        )

    starting_fx = float(required_fx.iloc[0])
    balance_usd = initial_reserve / starting_fx
    rows: list[dict] = []
    depletion_date: pd.Timestamp | None = None

    for date in required.index[1:]:
        previous_price = required.loc[date - pd.offsets.MonthEnd(1)]
        monthly_return = required.loc[date] / previous_price - 1
        fx_rate = float(required_fx.loc[date])
        before_withdrawal_usd = balance_usd * (1 + monthly_return)
        requested_withdrawal_usd = monthly_withdrawal / fx_rate
        actual_withdrawal_usd = min(
            requested_withdrawal_usd, max(before_withdrawal_usd, 0.0)
        )
        actual_withdrawal_krw = actual_withdrawal_usd * fx_rate
        balance_usd = max(before_withdrawal_usd - actual_withdrawal_usd, 0.0)
        ending_balance_krw = balance_usd * fx_rate

        if balance_usd <= 0:
            depletion_date = date

        rows.append(
            {
                "start_year": start_year,
                "date": date,
                "usdkrw": fx_rate,
                "monthly_return": monthly_return,
                "balance_before_withdrawal_usd": before_withdrawal_usd,
                "requested_withdrawal_krw": monthly_withdrawal,
                "withdrawal_usd": actual_withdrawal_usd,
                "withdrawal_krw": actual_withdrawal_krw,
                "ending_balance_usd": balance_usd,
                "ending_balance_krw": ending_balance_krw,
            }
        )

        if balance_usd <= 0:
            break

    result = {
        "start_year": start_year,
        "depleted_within_period": depletion_date is not None,
        "depletion_month": depletion_date.strftime("%Y-%m") if depletion_date else "-",
        "months_survived": len(rows),
        "starting_fx": starting_fx,
        "ending_fx": float(required_fx.loc[rows[-1]["date"]]),
        "ending_balance": rows[-1]["ending_balance_krw"],
        "total_withdrawn": sum(row["withdrawal_krw"] for row in rows),
    }
    return result, pd.DataFrame(rows)


def calculate_backtest_results(
    prices: pd.Series,
    fx_rates: pd.Series,
    first_start_year: int,
    last_start_year: int,
    years: int,
    initial_reserve: float,
    monthly_withdrawal: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """모든 시작연도의 요약 결과와 월별 상세 내역을 계산한다."""
    summaries: list[dict] = []
    details: list[pd.DataFrame] = []

    for start_year in range(first_start_year, last_start_year + 1):
        summary, detail = simulate_one_start(
            prices, fx_rates, start_year, initial_reserve, monthly_withdrawal, years
        )
        summaries.append(summary)
        details.append(detail)

    return pd.DataFrame(summaries), pd.concat(details, ignore_index=True)


# =============================================================================
# 3. 그래프 작성
# =============================================================================

BLUE = "#2F7DD3"
ORANGE = "#F06432"
TEXT_COLOR = "#0B0B0B"
MUTED_TEXT_COLOR = "#666666"
GRID_COLOR = "#DEDCD6"
BACKGROUND_COLOR = "#FFFFFF"
TICK_COLOR = "#777777"
FOOTNOTE_COLOR = "#777777"
REFERENCE_COLOR = "#F59E0B"
FONT_CANDIDATES = (
    "Pretendard",
    "Apple SD Gothic Neo",
    "Noto Sans CJK KR",
    "Malgun Gothic",
)
COLAB_FONT_PATH = Path("/content/.fonts/Pretendard-Regular.otf")
COLAB_FONT_URL = (
    "https://raw.githubusercontent.com/orioncactus/pretendard/main/"
    "packages/pretendard/dist/public/static/Pretendard-Regular.otf"
)


def configure_korean_font() -> None:
    """사용 가능한 한글 글꼴을 Matplotlib에 설정한다."""
    installed = {font.name for font in font_manager.fontManager.ttflist}
    font_name = next((name for name in FONT_CANDIDATES if name in installed), None)
    if font_name is None and IS_COLAB:
        COLAB_FONT_PATH.parent.mkdir(parents=True, exist_ok=True)
        if not COLAB_FONT_PATH.exists():
            urlretrieve(COLAB_FONT_URL, COLAB_FONT_PATH)
        font_manager.fontManager.addfont(COLAB_FONT_PATH)
        font_name = font_manager.FontProperties(fname=COLAB_FONT_PATH).get_name()
    if font_name is not None:
        plt.rcParams["font.family"] = font_name
    plt.rcParams["axes.unicode_minus"] = False


def draw_rounded_bars(
    ax: Axes,
    x_positions: list[int],
    amounts_in_100m: pd.Series,
    depleted_flags: pd.Series,
) -> list[FancyBboxPatch]:
    """잔액은 파란 막대, 고갈은 주황색 사선 막대로 표시한다."""
    base_bars = ax.bar(x_positions, amounts_in_100m, width=0.62, color="none")
    rounded_bars: list[FancyBboxPatch] = []
    for bar, value, depleted in zip(
        base_bars, amounts_in_100m, depleted_flags
    ):
        height = 0.035 if depleted else float(value)
        rounded = FancyBboxPatch(
            (bar.get_x(), 0),
            bar.get_width(),
            height,
            boxstyle="round,pad=0,rounding_size=0.055",
            linewidth=1.5 if depleted else 0,
            edgecolor=ORANGE if depleted else BLUE,
            facecolor="#FFF4EF" if depleted else BLUE,
            hatch="///" if depleted else None,
            zorder=3,
        )
        ax.add_patch(rounded)
        rounded_bars.append(rounded)
    for bar in base_bars:
        bar.remove()
    return rounded_bars


def add_chart_header(fig: Figure) -> None:
    """제목, 조건 설명과 범례를 추가한다."""
    fig.text(
        0.055,
        0.97,
        f"성장 준비금 {INVESTMENT_YEARS}년 생존 검증 | {TICKER} 시작연도별 백테스트",
        fontsize=21,
        color=TEXT_COLOR,
        fontweight="semibold",
        ha="left",
        va="top",
    )
    fig.text(
        0.055,
        0.915,
        f"초기 준비금 {format_korean_won(INITIAL_RESERVE_KRW)} · "
        f"매월 말 {format_korean_won(MONTHLY_WITHDRAWAL_KRW)} 인출 · 실제 원/달러 환율 반영",
        fontsize=15.5,
        color=MUTED_TEXT_COLOR,
        ha="left",
        va="top",
    )
    legend_items = [
        Patch(facecolor=BLUE, edgecolor=BLUE, label=f"{INVESTMENT_YEARS}년 후 잔액"),
        Patch(facecolor="#FFF4EF", edgecolor=ORANGE, hatch="///", label="고갈"),
    ]
    fig.legend(
        handles=legend_items,
        loc="upper left",
        bbox_to_anchor=(0.055, 0.835),
        frameon=False,
        ncol=2,
        fontsize=11.5,
        handlelength=1.0,
        columnspacing=1.5,
    )


def style_chart_axes(ax: Axes, summary: pd.DataFrame) -> None:
    """축, 눈금, 그리드와 여백을 정리한다."""
    x_positions = list(range(len(summary)))
    ax.axhline(
        INITIAL_RESERVE_KRW / WON_PER_EOK,
        color=REFERENCE_COLOR,
        linewidth=1.2,
        linestyle=(0, (4, 4)),
        alpha=0.38,
        zorder=1,
    )
    ax.text(
        0.995,
        1.0,
        f"시작 준비금 {format_korean_won(INITIAL_RESERVE_KRW)}",
        transform=ax.get_yaxis_transform(),
        ha="right",
        va="bottom",
        fontsize=12,
        color=REFERENCE_COLOR,
        alpha=0.88,
        bbox={"facecolor": BACKGROUND_COLOR, "edgecolor": "none", "pad": 1.5},
        zorder=5,
    )
    ax.set_xlabel("투자 시작연도", color=TEXT_COLOR, fontsize=13.5, labelpad=12)
    ax.set_ylabel(
        f"{INVESTMENT_YEARS}년 후 잔액(억원)",
        color=TICK_COLOR,
        fontsize=13.5,
        labelpad=10,
    )
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.1f}"))
    ax.set_xticks(x_positions, summary["start_year"].astype(str))
    ax.tick_params(axis="both", colors=TICK_COLOR, labelsize=12, length=0)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.9, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.margins(x=0.025)


def add_bar_labels(
    ax: Axes, rounded_bars: list[FancyBboxPatch], summary: pd.DataFrame
) -> None:
    """각 막대 위에 잔액 또는 고갈 표시를 붙인다."""
    for bar, value, depleted, months_survived in zip(
        rounded_bars,
        summary["ending_balance"],
        summary["depleted_within_period"],
        summary["months_survived"],
    ):
        if depleted:
            years, months = divmod(int(months_survived), 12)
            label = f"고갈\n({years}년 {months}개월)"
        else:
            label = f"{value / 100_000_000:.2f}억"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.025,
            label,
            ha="center",
            va="bottom",
            fontsize=13,
            color=ORANGE if depleted else TEXT_COLOR,
            fontweight="semibold",
        )



def save_chart(summary: pd.DataFrame, output_path: Path) -> None:
    """요약 결과를 참고 이미지 스타일의 PNG 막대그래프로 저장한다."""
    configure_korean_font()
    amounts_in_100m = summary["ending_balance"] / 100_000_000
    x_positions = list(range(len(summary)))

    fig, ax = plt.subplots(figsize=(12.5, 7.0), facecolor=BACKGROUND_COLOR)
    ax.set_facecolor(BACKGROUND_COLOR)

    rounded_bars = draw_rounded_bars(
        ax, x_positions, amounts_in_100m, summary["depleted_within_period"]
    )
    add_chart_header(fig)
    style_chart_axes(ax, summary)
    add_bar_labels(ax, rounded_bars, summary)

    fig.text(
        0.055, 0.995, "대도시 연구실",
        ha="left", va="top", fontsize=9, color=FOOTNOTE_COLOR,
    )
    fig.text(
        0.01,
        0.01,
        f"{TICKER} 배당·분할 및 원/달러 환율 반영\n※ 세금·수수료·물가상승률 제외",
        fontsize=13,
        color=FOOTNOTE_COLOR,
    )
    fig.tight_layout(rect=(0, 0.04, 1, 0.77))
    fig.savefig(
        output_path, dpi=200, bbox_inches="tight", facecolor=BACKGROUND_COLOR
    )
    plt.close(fig)


# =============================================================================
# 4. 결과 저장 및 화면 출력
# =============================================================================

SUMMARY_CSV_COLUMNS = {
    "start_year": "시작연도",
    "depleted_within_period": "기간내고갈여부",
    "depletion_month": "고갈월",
    "months_survived": "유지개월수",
    "starting_fx": "시작환율(원/달러)",
    "ending_fx": "종료환율(원/달러)",
    "ending_balance": "최종잔액(원)",
    "total_withdrawn": "총인출액(원)",
}

DETAIL_CSV_COLUMNS = {
    "start_year": "시작연도",
    "date": "날짜",
    "usdkrw": "원달러환율",
    "monthly_return": "월수익률",
    "balance_before_withdrawal_usd": "인출전잔액(달러)",
    "requested_withdrawal_krw": "요청인출액(원)",
    "withdrawal_usd": "실제인출액(달러)",
    "withdrawal_krw": "실제인출액(원)",
    "ending_balance_usd": "월말잔액(달러)",
    "ending_balance_krw": "월말잔액(원)",
}


def format_duration(months: int) -> str:
    """개월 수를 블로그 표에 적합한 연/개월 문자열로 변환한다."""
    years, remaining_months = divmod(months, 12)
    if remaining_months == 0:
        return f"{years}년"
    if years == 0:
        return f"{remaining_months}개월"
    return f"{years}년 {remaining_months}개월"


def format_korean_won(amount: float) -> str:
    """원화 금액을 반올림해 억원/만원 단위의 짧은 문자열로 변환한다."""
    rounded_manwon = int(amount / WON_PER_MANWON + 0.5)
    if rounded_manwon == 0:
        return "0원"

    eok, manwon = divmod(rounded_manwon, WON_PER_EOK // WON_PER_MANWON)
    if eok and manwon:
        return f"{eok:,}억 {manwon:,}만원"
    if eok:
        return f"{eok:,}억원"
    return f"{manwon:,}만원"


def save_results(
    summary: pd.DataFrame, detail: pd.DataFrame, output_dir: Path
) -> tuple[Path, Path, Path]:
    """요약 CSV, 월별 상세 CSV와 PNG 그래프를 저장한다."""
    output_dir.mkdir(parents=True, exist_ok=True)
    prefix = f"{TICKER.lower()}_{INVESTMENT_YEARS}y"
    summary_path = output_dir / f"{prefix}_summary.csv"
    detail_path = output_dir / f"{prefix}_monthly_detail.csv"
    chart_path = output_dir / f"{prefix}_ending_reserve.png"

    summary.rename(columns=SUMMARY_CSV_COLUMNS).to_csv(
        summary_path, index=False, encoding="utf-8-sig"
    )
    detail.rename(columns=DETAIL_CSV_COLUMNS).to_csv(
        detail_path, index=False, encoding="utf-8-sig"
    )
    save_chart(summary, chart_path)
    return summary_path, detail_path, chart_path


# =============================================================================
# 5. 실행 설정 및 진입점
# =============================================================================


def validate_parameters() -> None:
    """Colab 입력값을 검증한다."""
    if not TICKER:
        raise ValueError("투자종목을 입력하세요.")
    if FIRST_START_YEAR > LAST_START_YEAR:
        raise ValueError("최초 시작연도는 마지막 시작연도보다 늦을 수 없습니다.")
    if FIRST_START_YEAR < 2000:
        raise ValueError("QQQ 데이터 범위를 고려해 시작연도는 2000년 이후로 입력하세요.")
    if INVESTMENT_YEARS <= 0 or INITIAL_RESERVE_KRW <= 0:
        raise ValueError("투자기간과 초기 준비금은 0보다 커야 합니다.")
    if MONTHLY_WITHDRAWAL_KRW < 0:
        raise ValueError("월 인출액은 0 이상이어야 합니다.")


def summary_table_html(summary: pd.DataFrame) -> str:
    """시작연도별 핵심 결과를 스타일 가이드에 맞춘 HTML 표로 만든다."""
    rows = []
    for row in summary.itertuples(index=False):
        depleted = bool(row.depleted_within_period)
        result = "✕ 고갈" if depleted else "✓ 유지"
        depletion = (
            f"{format_duration(int(row.months_survived))} 후"
            if depleted else "-"
        )
        rows.append({
            "시작연도": int(row.start_year),
            "결과": result,
            "고갈 시점": depletion,
            "총 인출액": format_korean_won(float(row.total_withdrawn)),
            "최종 잔액": format_korean_won(float(row.ending_balance)),
        })
    table = pd.DataFrame(rows).to_html(
        index=False, border=0, classes="result-table summary-table"
    )
    css = """
    <style>
    .result-table-wrap {max-width:700px; margin:12px 0 28px; overflow-x:auto;
      border:1px solid #f0f2f5; border-radius:12px;}
    .result-table {width:100%; border-collapse:separate; border-spacing:0;
      font-family:Pretendard,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;
      font-size:13px; font-variant-numeric:tabular-nums; color:#1e293b;}
    .result-table thead th {padding:8px 12px; background:#2b4a75;
      border-bottom:2px solid #7fb3d5; color:white; font-weight:700;
      text-align:right; white-space:nowrap;}
    .result-table tbody td {padding:7px 12px; border-bottom:1px solid #f0f2f5;
      background:white; text-align:right; white-space:nowrap;}
    .result-table th:first-child,.result-table td:first-child {text-align:center;font-weight:600;}
    .result-table tbody tr:nth-child(even) td {background:#fafbfc;}
    .result-table tbody tr:hover td {background:#eff6ff;}
    .result-table tbody tr:last-child td {border-bottom:0;}
    </style>
    """
    return css + f'<div class="result-table-wrap">{table}</div>'


def display_download_button(label: str, path: Path) -> None:
    """Colab에서는 다운로드 버튼, 로컬에서는 파일 링크를 표시한다."""
    from IPython.display import FileLink, display

    if IS_COLAB:
        import ipywidgets as widgets
        from google.colab import files

        button = widgets.Button(
            description=f"{label} 다운로드",
            icon="download",
            tooltip=path.name,
        )
        button.on_click(lambda _: files.download(str(path)))
        display(button)
    else:
        display(FileLink(str(path)))


def display_results(
    summary: pd.DataFrame,
    result_paths: tuple[Path, Path, Path],
) -> None:
    """실행 조건, 요약표, 그래프와 결과 파일 링크를 표시한다."""
    from IPython.display import HTML, Image, Markdown, display

    chart_path = result_paths[-1]
    display(Markdown(
        "## 실행 조건\n"
        f"- 투자종목: **{TICKER}**\n"
        f"- 시작연도: **{FIRST_START_YEAR}~{LAST_START_YEAR}년**\n"
        f"- 투자기간: **{INVESTMENT_YEARS}년**\n"
        f"- 초기 준비금: **{format_korean_won(INITIAL_RESERVE_KRW)}**\n"
        f"- 월 인출액: **{format_korean_won(MONTHLY_WITHDRAWAL_KRW)}**"
    ))
    display(Markdown("## 시작연도별 결과"))
    display(HTML(summary_table_html(summary)))
    display(Markdown("## 종료 시점 잔액 그래프"))
    display(Image(filename=str(chart_path)))
    display(Markdown("## 결과 파일 다운로드"))
    labels = ("요약 CSV", "월별 상세 CSV", "그래프 PNG")
    for label, path in zip(labels, result_paths):
        display_download_button(label, path)


def main() -> None:
    validate_parameters()
    prices, fx_rates = load_market_data(
        ticker=TICKER,
        first_start_year=FIRST_START_YEAR,
        last_start_year=LAST_START_YEAR,
        years=INVESTMENT_YEARS,
    )
    summary, detail = calculate_backtest_results(
        prices=prices,
        fx_rates=fx_rates,
        first_start_year=FIRST_START_YEAR,
        last_start_year=LAST_START_YEAR,
        years=INVESTMENT_YEARS,
        initial_reserve=INITIAL_RESERVE_KRW,
        monthly_withdrawal=MONTHLY_WITHDRAWAL_KRW,
    )
    result_paths = save_results(summary, detail, OUTPUT_DIR)
    display_results(summary, result_paths)


main()
